# Raw ROOTを経由する二段階の解析

第1段階で生データをデコード・同期して保存し、第2段階で積分・較正・飛跡・イベント再構成を行います。ゲインやドリフト速度を更新したときは第2段階だけ再実行します。
TPCだけなら `smile3_tpc_raw` → `smile3_tpc_only`、TPC＋PSAなら `smile3_etcc_raw` → `smile3_reconstruct` です。

In [ ]:
from pathlib import Path
import sys, subprocess
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_script/run_chain.py').exists())
RUN_CHAIN = REPO / 'run_script/run_chain.py'
ETCC = True  # True: TPC＋PSA、False: TPCのみ
RUN_DIR = Path('/path/to/storage_g0/<書き換え>')
EXTRA_DIR = Path('/path/to/storage_g1/<書き換え>')
OUT = Path('/path/to/analysis/<書き換え>')
RAW_ROOT = OUT / 'raw.root'
RECON_ROOT = OUT / 'reconstructed.root'
N_EVENTS = 10000  # 動作確認のイベント数 [events]。全件なら -1


## 第1段階：生情報をROOTへ保存

ここは同じrunについて一度実行します。再実行による上書きを防ぐため、既に出力があれば止めます。保存時に情報の選別は行いません。

In [ ]:
if any('<書き換え>' in str(p) for p in (RUN_DIR, EXTRA_DIR, OUT)):
    raise ValueError('自分の環境の入力・出力パスを指定してください')
if RAW_ROOT.exists():
    raise FileExistsError(f'{RAW_ROOT}: 作成済みならこのセルを飛ばして第2段階へ進んでください')
subprocess.run([sys.executable, str(RUN_CHAIN), 'smile3_etcc_raw' if ETCC else 'smile3_tpc_raw',
                '--data-dir', str(RUN_DIR), '--extra-data-dir', str(EXTRA_DIR),
                '-n', str(N_EVENTS), '-o', str(RAW_ROOT)], check=True)


## 第2段階：ROOTから再構成

生データのディレクトリは渡しません。較正ファイルや解析条件を変える場合は、この段階だけ繰り返します。ETCCにはPSAの生情報も保存したROOTが必要です。

In [ ]:
subprocess.run([sys.executable, str(RUN_CHAIN), 'smile3_reconstruct' if ETCC else 'smile3_tpc_only',
                '--root-input', str(RAW_ROOT), '-o', str(RECON_ROOT)], check=True)
print(RECON_ROOT)


## 較正・診断

同じRAW_ROOTをTPC_DriftVelocity.ipynb、TPC_GainMap.ipynb、TPC_FADC_Accumulation.ipynbのRAWSに指定します。測定条件が同じ複数runはRAWSに列挙して積算できます。
保存したROOTには第1段階で指定したイベント数だけが含まれます。最初の確認後、全件のROOTを別名で作る場合はN_EVENTS=-1を指定してください。